# Validación de datos - Proyecto OSIPTEL

El presente notebook tiene como finalidad verificar la estructura,
disponibilidad y consistencia de los conjuntos de datos procesados
por el equipo de ingeniería de datos.

Fuente: Oracle Cloud Infrastructure
Rama: feature/oci-bronze
Commit: c5cc727
Versión Gold F1: v2_b6e4155ae58e8f9106d8
Versión curada: v1_17b651ac372a96fb18fa

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

spark = (
    SparkSession.builder
    .appName("OSIPTEL_Validacion_Datos")
    .master("local[2]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark iniciado correctamente")
print("Version de Spark:", spark.version)
print("Modo de ejecucion:", spark.sparkContext.master)

26/11/27 11:38:05 WARN Utils: Your hostname, debian resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/27 11:38:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/27 11:38:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark iniciado correctamente
Version de Spark: 3.5.1
Modo de ejecucion: local[2]


In [2]:
BASE = (
    next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "run_pipeline.py").is_file()
    and (p / "notebooks").is_dir()
)
    / "data"
    / "paquete_20261125_c5cc727"
)

print("Ruta:", BASE)
print("Carpeta disponible:", BASE.is_dir())

for carpeta in sorted(BASE.iterdir()):
    if carpeta.is_dir():
        print(carpeta.name)

Ruta: /home/nevado/BigData_UPAO/OSIPTEL_Analisis_Final/data/paquete_20261125_c5cc727
Carpeta disponible: True
comparacion_f1_f4
f1_historico_nacional
f1_historico_territorial
f2_menciones_verificadas
f3_cifras_contextualizadas
f4_conflictos_excluidos
f4_hogares_verificados
f4_pendientes


In [3]:
esperados = {
    "f1_historico_nacional": 71,
    "f1_historico_territorial": 129225,
    "f2_menciones_verificadas": 287,
    "f3_cifras_contextualizadas": 12,
    "f4_hogares_verificados": 20,
    "f4_pendientes": 74,
    "f4_conflictos_excluidos": 102,
    "comparacion_f1_f4": 5
}

datasets = {}
validacion = []

for nombre, esperado in esperados.items():

    ruta = (BASE / nombre).as_uri()
    df = spark.read.parquet(ruta)

    datasets[nombre] = df
    obtenido = df.count()

    validacion.append({
        "Dataset": nombre,
        "Filas esperadas": esperado,
        "Filas obtenidas": obtenido,
        "Columnas": len(df.columns),
        "Estado": "OK" if obtenido == esperado else "REVISAR"
    })

pd.DataFrame(validacion)

,Dataset,Filas esperadas,Filas obtenidas,Columnas,Estado
0,f1_historico_nacional,71,71,6,OK
1,f1_historico_territorial,129225,129225,11,OK
2,f2_menciones_verificadas,287,287,15,OK
3,f3_cifras_contextualizadas,12,12,17,OK
4,f4_hogares_verificados,20,20,13,OK
5,f4_pendientes,74,74,14,OK
6,f4_conflictos_excluidos,102,102,8,OK
7,comparacion_f1_f4,5,5,13,OK


In [4]:
f1_nacional = datasets["f1_historico_nacional"]
f1_territorial = datasets["f1_historico_territorial"]
f2 = datasets["f2_menciones_verificadas"]
f3 = datasets["f3_cifras_contextualizadas"]
f4 = datasets["f4_hogares_verificados"]
comparacion = datasets["comparacion_f1_f4"]

In [5]:
for nombre, df in datasets.items():
    print("=" * 60)
    print(nombre)
    df.printSchema()

f1_historico_nacional
root
 |-- periodo: string (nullable = true)
 |-- conexiones_fijas: long (nullable = true)
 |-- filas_unicas: long (nullable = true)
 |-- filas_sin_departamento: long (nullable = true)
 |-- origen_sha256: string (nullable = true)
 |-- regla_version: string (nullable = true)

f1_historico_territorial
root
 |-- periodo: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- operador: string (nullable = true)
 |-- tecnologia: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- rango_velocidad: string (nullable = true)
 |-- conexiones_fijas: long (nullable = true)
 |-- filas_origen: long (nullable = true)
 |-- filas_observadas: long (nullable = true)
 |-- origen_sha256: string (nullable = true)
 |-- regla_version: string (nullable = true)

f2_menciones_verificadas
root
 |-- documento: string (nullable = true)
 |-- sha256: string (nullable = true)
 |-- pagina_pdf: string (nullable = true)
 |-- linea_pdf: string (nullable = true)
 |--

In [6]:
print("F1 - HISTORICO NACIONAL")
display(f1_nacional.orderBy(F.desc("periodo")).limit(10).toPandas())

print("F1 - HISTORICO TERRITORIAL")
display(f1_territorial.limit(10).toPandas())

print("F4 - HOGARES CON INTERNET")
display(f4.limit(10).toPandas())

print("COMPARACION F1 - F4")
display(comparacion.toPandas())

F1 - HISTORICO NACIONAL


,periodo,conexiones_fijas,filas_unicas,filas_sin_departamento,origen_sha256,regla_version
0,2026-03,4135995,4531,0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
1,2025-12,4379544,4750,0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
2,2025-09,4289604,4695,0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
3,2025-06,4194402,4683,0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
4,2025-03,4098323,4704,0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
5,2024-12,4014572,4793,0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
6,2024-09,3952448,4819,0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
7,2024-06,3800822,4729,0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
8,2024-03,3669365,4885,0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
9,2023-12,3563247,4814,0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...


F1 - HISTORICO TERRITORIAL


,periodo,departamento,operador,tecnologia,segmento,rango_velocidad,conexiones_fijas,filas_origen,filas_observadas,origen_sha256,regla_version
0,2012-12,LIMA Y CALLAO,ENTEL PERU S.A.,4) WIMAX,RESIDENCIAL,NO DECLARADO,2792,4,4,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
1,2012-12,LIMA Y CALLAO,GILAT TO HOME PERU S.A.,7) OTRAS TECNOLOGÍAS,COMERCIAL,NO DECLARADO,28,4,4,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
2,2012-12,OTRAS REGIONES,GILAT TO HOME PERU S.A.,7) OTRAS TECNOLOGÍAS,COMERCIAL,NO DECLARADO,413,5,5,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
3,2013-03,OTRAS REGIONES,AMERICA MOVIL PERU S.A.C.,1) DIAL-UP FIJO,RESIDENCIAL,NO DECLARADO,210,1,1,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
4,2013-03,OTRAS REGIONES,COMPAÑIA DE COMERCIO GLOBAL S.A.C.,7) OTRAS TECNOLOGÍAS,COMERCIAL,NO DECLARADO,31,4,4,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
5,2013-03,OTRAS REGIONES,STAR GLOBAL COM S.A.C.,3) CABLEMODEM,RESIDENCIAL,NO DECLARADO,7055,5,5,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
6,2013-03,OTRAS REGIONES,TELEFONICA DEL PERU S.A.A.,2) XDSL,RESIDENCIAL,NO DECLARADO,404592,5,5,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
7,2013-03,OTRAS REGIONES,TERRA NETWORKS PERU S.A.C.,1) DIAL-UP FIJO,COMERCIAL,NO DECLARADO,63,1,1,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
8,2013-03,OTRAS REGIONES,TERRA NETWORKS PERU S.A.C.,1) DIAL-UP FIJO,RESIDENCIAL,NO DECLARADO,40,1,1,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...
9,2013-03,OTRAS REGIONES,WINNER SYSTEMS S.A.C.,1) DIAL-UP FIJO,RESIDENCIAL,NO DECLARADO,139,1,1,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,osiptel-historico-f1-v2-dedup7-depto-separado-...


F4 - HOGARES CON INTERNET


,periodo,ambito,indicador,porcentaje,unidad,pdf,pagina_pdf,sha256_pdf,evidencia_periodo_y_area,estado_revision,uso_permitido,estado_curaduria,aprobacion_institucional
0,2026-T2,Nacional,Hogares con acceso a Internet,64.5,porcentaje_hogares,boletin-tic-iit2026.pdf,6,bc1cf662ff64098e4e368b3e0bf2377c29bfb536164330...,SI,VERIFICADO_EN_NARRATIVA_PDF,COMPARACION_DESCRIPTIVA_TRIMESTRAL; NO_EQUIVAL...,VERIFICADO_EN_NARRATIVA_PDF,NO_CONSTA
1,2026-T2,Lima Metropolitana,Hogares con acceso a Internet,84.7,porcentaje_hogares,boletin-tic-iit2026.pdf,6,bc1cf662ff64098e4e368b3e0bf2377c29bfb536164330...,SI,VERIFICADO_EN_NARRATIVA_PDF,COMPARACION_DESCRIPTIVA_TRIMESTRAL; NO_EQUIVAL...,VERIFICADO_EN_NARRATIVA_PDF,NO_CONSTA
2,2026-T2,Resto urbano,Hogares con acceso a Internet,66.2,porcentaje_hogares,boletin-tic-iit2026.pdf,6,bc1cf662ff64098e4e368b3e0bf2377c29bfb536164330...,SI,VERIFICADO_EN_NARRATIVA_PDF,COMPARACION_DESCRIPTIVA_TRIMESTRAL; NO_EQUIVAL...,VERIFICADO_EN_NARRATIVA_PDF,NO_CONSTA
3,2026-T2,Area rural,Hogares con acceso a Internet,27.1,porcentaje_hogares,boletin-tic-iit2026.pdf,6,bc1cf662ff64098e4e368b3e0bf2377c29bfb536164330...,SI,VERIFICADO_EN_NARRATIVA_PDF,COMPARACION_DESCRIPTIVA_TRIMESTRAL; NO_EQUIVAL...,VERIFICADO_EN_NARRATIVA_PDF,NO_CONSTA
4,2025-T4,Nacional,Hogares con acceso a Internet,60.1,porcentaje_hogares,boletin-tic-oct_dic2025.pdf,6,ee51c138f73def9256f917be9b8c632ec0f817d86a4d4f...,SI,VERIFICADO_EN_NARRATIVA_PDF,COMPARACION_DESCRIPTIVA_TRIMESTRAL; NO_EQUIVAL...,VERIFICADO_EN_NARRATIVA_PDF,NO_CONSTA
5,2025-T4,Lima Metropolitana,Hogares con acceso a Internet,78.2,porcentaje_hogares,boletin-tic-oct_dic2025.pdf,6,ee51c138f73def9256f917be9b8c632ec0f817d86a4d4f...,SI,VERIFICADO_EN_NARRATIVA_PDF,COMPARACION_DESCRIPTIVA_TRIMESTRAL; NO_EQUIVAL...,VERIFICADO_EN_NARRATIVA_PDF,NO_CONSTA
6,2025-T4,Resto urbano,Hogares con acceso a Internet,63.1,porcentaje_hogares,boletin-tic-oct_dic2025.pdf,6,ee51c138f73def9256f917be9b8c632ec0f817d86a4d4f...,SI,VERIFICADO_EN_NARRATIVA_PDF,COMPARACION_DESCRIPTIVA_TRIMESTRAL; NO_EQUIVAL...,VERIFICADO_EN_NARRATIVA_PDF,NO_CONSTA
7,2025-T4,Area rural,Hogares con acceso a Internet,23.4,porcentaje_hogares,boletin-tic-oct_dic2025.pdf,6,ee51c138f73def9256f917be9b8c632ec0f817d86a4d4f...,SI,VERIFICADO_EN_NARRATIVA_PDF,COMPARACION_DESCRIPTIVA_TRIMESTRAL; NO_EQUIVAL...,VERIFICADO_EN_NARRATIVA_PDF,NO_CONSTA
8,2026-T1,Nacional,Hogares con acceso a Internet,62.7,porcentaje_hogares,boletin_tic_itrim2026.pdf,6,23f24fc00cbfd4a6c259bc33cc8bbdee8f1d774a36805b...,SI,VERIFICADO_EN_NARRATIVA_PDF,COMPARACION_DESCRIPTIVA_TRIMESTRAL; NO_EQUIVAL...,VERIFICADO_EN_NARRATIVA_PDF,NO_CONSTA
9,2026-T1,Lima Metropolitana,Hogares con acceso a Internet,82.7,porcentaje_hogares,boletin_tic_itrim2026.pdf,6,23f24fc00cbfd4a6c259bc33cc8bbdee8f1d774a36805b...,SI,VERIFICADO_EN_NARRATIVA_PDF,COMPARACION_DESCRIPTIVA_TRIMESTRAL; NO_EQUIVAL...,VERIFICADO_EN_NARRATIVA_PDF,NO_CONSTA


COMPARACION F1 - F4


,periodo_trimestre,hogares_internet_pct_f4,pdf_inei,pagina_pdf_inei,sha256_pdf_inei,periodo_cierre_f1,conexiones_fijas_f1,sha256_excel_f1,estado_cobertura,tipo_comparacion,nota,estado_curaduria,aprobacion_institucional
0,2026-T2,64.5,boletin-tic-iit2026.pdf,6,bc1cf662ff64098e4e368b3e0bf2377c29bfb536164330...,None,NaN,None,SOLO_INEI,DESCRIPTIVA_NO_CAUSAL,OSIPTEL: numero de conexiones fijas. INEI: por...,INDICADOR_F4_VERIFICADO_EN_PDF_Y_GOLD_F1_VERIF...,NO_CONSTA
1,2025-T4,60.1,boletin-tic-oct_dic2025.pdf,6,ee51c138f73def9256f917be9b8c632ec0f817d86a4d4f...,2025-12,4379544.0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,AMBAS_FUENTES,DESCRIPTIVA_NO_CAUSAL,OSIPTEL: numero de conexiones fijas. INEI: por...,INDICADOR_F4_VERIFICADO_EN_PDF_Y_GOLD_F1_VERIF...,NO_CONSTA
2,2026-T1,62.7,boletin_tic_itrim2026.pdf,6,23f24fc00cbfd4a6c259bc33cc8bbdee8f1d774a36805b...,2026-03,4135995.0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,AMBAS_FUENTES,DESCRIPTIVA_NO_CAUSAL,OSIPTEL: numero de conexiones fijas. INEI: por...,INDICADOR_F4_VERIFICADO_EN_PDF_Y_GOLD_F1_VERIF...,NO_CONSTA
3,2025-T1,58.9,informe-tecnico_tecnologiasdelainformacion_ene...,6,86a57b723ccef2d93a2c96f5a9b988a4a015bf12fd03c2...,2025-03,4098323.0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,AMBAS_FUENTES,DESCRIPTIVA_NO_CAUSAL,OSIPTEL: numero de conexiones fijas. INEI: por...,INDICADOR_F4_VERIFICADO_EN_PDF_Y_GOLD_F1_VERIF...,NO_CONSTA
4,2025-T3,59.7,tic-iii-trimestre_2025_1.pdf,6,e84e8a7dae81f52259417228ebdbb360135fd2d9a217e8...,2025-09,4289604.0,e6e88de8b58ab84f25159d63584300e453d9bb7fe5057f...,AMBAS_FUENTES,DESCRIPTIVA_NO_CAUSAL,OSIPTEL: numero de conexiones fijas. INEI: por...,INDICADOR_F4_VERIFICADO_EN_PDF_Y_GOLD_F1_VERIF...,NO_CONSTA


In [7]:
duplicados = []

for nombre, df in datasets.items():

    total = df.count()
    unicos = df.distinct().count()

    duplicados.append({
        "Dataset": nombre,
        "Total de filas": total,
        "Filas unicas": unicos,
        "Duplicados exactos": total - unicos
    })

display(pd.DataFrame(duplicados))

,Dataset,Total de filas,Filas unicas,Duplicados exactos
0,f1_historico_nacional,71,71,0
1,f1_historico_territorial,129225,129225,0
2,f2_menciones_verificadas,287,287,0
3,f3_cifras_contextualizadas,12,12,0
4,f4_hogares_verificados,20,20,0
5,f4_pendientes,74,74,0
6,f4_conflictos_excluidos,102,102,0
7,comparacion_f1_f4,5,5,0


In [8]:
periodos = (
    f1_nacional
    .select("periodo")
    .distinct()
    .orderBy("periodo")
    .toPandas()
)

periodos["fecha"] = pd.to_datetime(
    periodos["periodo"],
    format="%Y-%m"
)

periodos["meses_desde_anterior"] = (
    periodos["fecha"].dt.year * 12
    + periodos["fecha"].dt.month
).diff()

print("Primer periodo:", periodos["periodo"].min())
print("Ultimo periodo:", periodos["periodo"].max())
print("Periodos disponibles:", len(periodos))

display(periodos)

Primer periodo: 1999-12
Ultimo periodo: 2026-03
Periodos disponibles: 71


,periodo,fecha,meses_desde_anterior
0,1999-12,1999-12-01,NaN
1,2000-12,2000-12-01,12.0
2,2001-12,2001-12-01,12.0
3,2002-12,2002-12-01,12.0
4,2003-12,2003-12-01,12.0
...,...,...,...
66,2025-03,2025-03-01,3.0
67,2025-06,2025-06-01,3.0
68,2025-09,2025-09-01,3.0
69,2025-12,2025-12-01,3.0


In [9]:
print("DISTRIBUCION DE INTERVALOS TEMPORALES")

display(
    periodos["meses_desde_anterior"]
    .value_counts()
    .sort_index()
    .rename_axis("Intervalo en meses")
    .reset_index(name="Cantidad")
)

print("PERIODOS CON INTERVALOS DIFERENTES A 3 MESES")

display(
    periodos[
        periodos["meses_desde_anterior"].notna()
        & (periodos["meses_desde_anterior"] != 3)
    ]
)

DISTRIBUCION DE INTERVALOS TEMPORALES


,Intervalo en meses,Cantidad
0,3.0,57
1,6.0,2
2,12.0,11


PERIODOS CON INTERVALOS DIFERENTES A 3 MESES


,periodo,fecha,meses_desde_anterior
1,2000-12,2000-12-01,12.0
2,2001-12,2001-12-01,12.0
3,2002-12,2002-12-01,12.0
4,2003-12,2003-12-01,12.0
5,2004-12,2004-12-01,12.0
6,2005-12,2005-12-01,12.0
7,2006-12,2006-12-01,12.0
8,2007-12,2007-12-01,12.0
9,2008-12,2008-12-01,12.0
10,2009-12,2009-12-01,12.0


In [10]:
geografia = (
    f1_territorial
    .groupBy("departamento")
    .agg(
        F.min("periodo").alias("primer_periodo"),
        F.max("periodo").alias("ultimo_periodo"),
        F.countDistinct("periodo").alias("periodos_disponibles"),
        F.count("*").alias("registros")
    )
    .orderBy("departamento")
)

display(geografia.toPandas())

,departamento,primer_periodo,ultimo_periodo,periodos_disponibles,registros
0,AMAZONAS,2014-12,2026-03,46,2517
1,ANCASH,2014-12,2026-03,46,5860
2,APURIMAC,2014-12,2026-03,46,3167
3,AREQUIPA,2014-12,2026-03,46,7269
4,AYACUCHO,2014-12,2026-03,46,3947
5,CAJAMARCA,2014-12,2026-03,46,4901
6,CALLAO,2014-12,2026-03,46,7003
7,CUSCO,2014-12,2026-03,46,6354
8,HUANCAVELICA,2014-12,2026-03,46,2713
9,HUANUCO,2014-12,2026-03,46,4412


In [11]:
cobertura_reportes = (
    f1_territorial
    .groupBy("periodo")
    .agg(
        F.countDistinct("departamento").alias("categorias_geograficas"),
        F.countDistinct("operador").alias("operadores_reportados"),
        F.countDistinct("tecnologia").alias("tecnologias_reportadas"),
        F.count("*").alias("registros")
    )
    .orderBy("periodo")
)

display(cobertura_reportes.toPandas())

,periodo,categorias_geograficas,operadores_reportados,tecnologias_reportadas,registros
0,2012-12,2,12,5,51
1,2013-03,2,12,5,53
2,2013-06,2,12,5,52
3,2013-09,2,11,5,49
4,2013-12,2,11,5,48
5,2014-03,2,11,5,47
6,2014-06,2,12,5,50
7,2014-09,2,12,5,48
8,2014-12,28,13,5,221
9,2015-03,28,13,5,241


In [12]:
claves = {
    "f1_historico_nacional": [
        "periodo"
    ],
    "f1_historico_territorial": [
        "periodo",
        "departamento",
        "operador",
        "tecnologia",
        "segmento",
        "rango_velocidad"
    ],
    "f4_hogares_verificados": [
        "periodo",
        "ambito",
        "indicador"
    ],
    "comparacion_f1_f4": [
        "periodo_trimestre"
    ]
}

control_claves = []

for nombre, columnas in claves.items():

    df = datasets[nombre]

    repetidos = (
        df.groupBy(*columnas)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    control_claves.append({
        "Dataset": nombre,
        "Columnas de la clave": len(columnas),
        "Claves repetidas": repetidos,
        "Estado": "OK" if repetidos == 0 else "REVISAR"
    })

display(pd.DataFrame(control_claves))

,Dataset,Columnas de la clave,Claves repetidas,Estado
0,f1_historico_nacional,1,0,OK
1,f1_historico_territorial,6,0,OK
2,f4_hogares_verificados,3,0,OK
3,comparacion_f1_f4,1,0,OK


In [13]:
territorial_total = (
    f1_territorial
    .groupBy("periodo")
    .agg(
        F.sum("conexiones_fijas")
        .alias("total_territorial")
    )
)

conciliacion = (
    f1_nacional
    .select(
        "periodo",
        "conexiones_fijas",
        "filas_sin_departamento"
    )
    .join(
        territorial_total,
        on="periodo",
        how="full"
    )
    .withColumn(
        "diferencia",
        F.col("conexiones_fijas") - F.col("total_territorial")
    )
    .orderBy("periodo")
)

display(conciliacion.toPandas())

,periodo,conexiones_fijas,filas_sin_departamento,total_territorial,diferencia
0,1999-12,120014,1,NaN,NaN
1,2000-12,131380,3,NaN,NaN
2,2001-12,150342,6,NaN,NaN
3,2002-12,238070,10,NaN,NaN
4,2003-12,303648,10,NaN,NaN
...,...,...,...,...,...
66,2025-03,4098323,0,4098323.0,0.0
67,2025-06,4194402,0,4194402.0,0.0
68,2025-09,4289604,0,4289604.0,0.0
69,2025-12,4379544,0,4379544.0,0.0


In [14]:
resumen_nulos = []

for nombre, df in datasets.items():

    conteos = df.select([
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in df.columns
    ]).first().asDict()

    for columna, cantidad in conteos.items():
        if cantidad > 0:
            resumen_nulos.append({
                "Dataset": nombre,
                "Columna": columna,
                "Valores nulos": cantidad
            })

display(
    pd.DataFrame(
        resumen_nulos,
        columns=["Dataset", "Columna", "Valores nulos"]
    )
)

,Dataset,Columna,Valores nulos
0,comparacion_f1_f4,periodo_cierre_f1,1
1,comparacion_f1_f4,conexiones_fijas_f1,1
2,comparacion_f1_f4,sha256_excel_f1,1


In [15]:
territorial_estudio = f1_territorial.filter(
    F.col("periodo") >= "2016-03"
)

departamentos_periodo = (
    territorial_estudio
    .groupBy("departamento")
    .agg(
        F.min("periodo").alias("primer_periodo"),
        F.max("periodo").alias("ultimo_periodo"),
        F.countDistinct("periodo").alias("trimestres"),
        F.count("*").alias("registros")
    )
    .orderBy("departamento")
)

display(departamentos_periodo.toPandas())

print(
    "Categorias geograficas:",
    territorial_estudio.select("departamento").distinct().count()
)

print(
    "Periodos disponibles:",
    territorial_estudio.select("periodo").distinct().count()
)

,departamento,primer_periodo,ultimo_periodo,trimestres,registros
0,AMAZONAS,2016-03,2026-03,41,2466
1,ANCASH,2016-03,2026-03,41,5745
2,APURIMAC,2016-03,2026-03,41,3108
3,AREQUIPA,2016-03,2026-03,41,7116
4,AYACUCHO,2016-03,2026-03,41,3891
5,CAJAMARCA,2016-03,2026-03,41,4816
6,CALLAO,2016-03,2026-03,41,6878
7,CUSCO,2016-03,2026-03,41,6226
8,HUANCAVELICA,2016-03,2026-03,41,2668
9,HUANUCO,2016-03,2026-03,41,4338


Categorias geograficas: 25
Periodos disponibles: 41


In [16]:
tecnologias = (
    f1_territorial
    .groupBy("tecnologia")
    .agg(
        F.min("periodo").alias("primer_periodo"),
        F.max("periodo").alias("ultimo_periodo"),
        F.countDistinct("periodo").alias("periodos"),
        F.count("*").alias("registros")
    )
    .orderBy("tecnologia")
)

display(tecnologias.toPandas())

,tecnologia,primer_periodo,ultimo_periodo,periodos,registros
0,1) DIAL-UP FIJO,2012-12,2018-06,23,124
1,2) XDSL,2012-12,2026-03,54,15634
2,3) CABLEMODEM,2012-12,2026-03,54,24501
3,4) WIMAX,2012-12,2026-03,54,2045
4,5) FIBRA ÓPTICA,2023-03,2026-03,13,35102
5,6) SATELITAL,2023-03,2026-03,13,5427
6,7) OTRAS TECNOLOGÍAS,2012-12,2026-03,54,46392


In [17]:
tecnologias_periodo = (
    f1_territorial
    .groupBy("periodo")
    .agg(
        F.sort_array(
            F.collect_set("tecnologia")
        ).alias("tecnologias")
    )
    .orderBy("periodo")
)

display(
    tecnologias_periodo.filter(
        F.col("periodo").isin(
            "2018-06",
            "2018-09",
            "2022-12",
            "2023-03",
            "2026-03"
        )
    ).toPandas()
)

,periodo,tecnologias
0,2018-06,"[1) DIAL-UP FIJO, 2) XDSL, 3) CABLEMODEM, 4) W..."
1,2018-09,"[2) XDSL, 3) CABLEMODEM, 4) WIMAX, 7) OTRAS TE..."
2,2022-12,"[2) XDSL, 3) CABLEMODEM, 4) WIMAX, 7) OTRAS TE..."
3,2023-03,"[2) XDSL, 3) CABLEMODEM, 4) WIMAX, 5) FIBRA ÓP..."
4,2026-03,"[2) XDSL, 3) CABLEMODEM, 4) WIMAX, 5) FIBRA ÓP..."


In [18]:
control_numerico = (
    f1_territorial
    .agg(
        F.min("conexiones_fijas").alias("minimo"),
        F.max("conexiones_fijas").alias("maximo"),
        F.count(
            F.when(F.col("conexiones_fijas") < 0, 1)
        ).alias("negativos"),
        F.count(
            F.when(F.col("conexiones_fijas") == 0, 1)
        ).alias("ceros")
    )
)

display(control_numerico.toPandas())

print("CONTROL DE PORCENTAJES INEI")

display(
    f4.agg(
        F.min("porcentaje").alias("minimo"),
        F.max("porcentaje").alias("maximo"),
        F.count(
            F.when(
                ~F.col("porcentaje").between(0, 100), 1
            )
        ).alias("fuera_de_rango")
    ).toPandas()
)

,minimo,maximo,negativos,ceros
0,0,735107,0,386


CONTROL DE PORCENTAJES INEI


,minimo,maximo,fuera_de_rango
0,20.5,84.7,0


In [20]:
conciliacion_estudio = conciliacion.filter(
    F.col("periodo") >= "2016-03"
)

display(
    conciliacion_estudio.agg(
        F.count("*").alias("periodos_evaluados"),
        F.count(
            F.when(
                F.col("diferencia") == 0, 1
            )
        ).alias("periodos_coincidentes"),
        F.count(
            F.when(
                F.col("diferencia") != 0, 1
            )
        ).alias("periodos_con_diferencias"),
        F.count(
            F.when(
                F.col("diferencia").isNull(), 1
            )
        ).alias("periodos_sin_conciliar")
    ).toPandas()
)

display(
    conciliacion_estudio.filter(
        (F.col("diferencia") != 0)
        | F.col("diferencia").isNull()
    ).toPandas()
)

,periodos_evaluados,periodos_coincidentes,periodos_con_diferencias,periodos_sin_conciliar
0,41,41,0,0


,periodo,conexiones_fijas,filas_sin_departamento,total_territorial,diferencia


## Conclusiones de la validación de datos

Se verificaron los ocho conjuntos de datos entregados, comprobando que la cantidad de registros coincide con la documentación y que no existen duplicados exactos ni claves de negocio repetidas en las tablas evaluadas.

Se identificaron tres valores nulos correspondientes a la ausencia de información de OSIPTEL para el segundo trimestre de 2026 en la tabla comparativa con el INEI. Estos valores se conservarán para evitar interpretaciones incorrectas.

El histórico nacional presenta diferentes frecuencias temporales. Sin embargo, desde marzo de 2016 hasta marzo de 2026 se dispone de 41 trimestres consecutivos con información de las mismas 25 unidades territoriales. Asimismo, los totales territoriales coinciden con los nacionales durante todo este intervalo.

Se identificó que la fibra óptica y la tecnología satelital aparecen como categorías explícitas a partir de marzo de 2023. Por ello, su análisis específico se limitará inicialmente al periodo comprendido entre 2023 y 2026.

No se encontraron cantidades negativas de conexiones ni porcentajes de acceso a internet fuera del rango permitido. Los registros con cero conexiones se conservarán, debido a que pueden representar observaciones válidas.

Los resultados permiten iniciar el análisis exploratorio, considerando las limitaciones temporales, geográficas y tecnológicas identificadas.